In [5]:
pip install scikit-learn xgboost nltk pandas numpy dvc PyYAML


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import re 
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/entbappy/Branching-tutorial/refs/heads/master/tweet_emotions.csv")

In [4]:
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [5]:
df.drop(columns=['tweet_id'], inplace=True)

In [6]:
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [7]:
df['sentiment'].unique()

<StringArray>
[     'empty',    'sadness', 'enthusiasm',    'neutral',      'worry',
   'surprise',       'love',        'fun',       'hate',  'happiness',
    'boredom',     'relief',      'anger']
Length: 13, dtype: str

In [8]:
df = df[df["sentiment"].isin(["sadness","happiness"])]

In [20]:
shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [21]:
shuffled_df.head()

,sentiment,content
0,sadness,looks like we're rained out for weekend climbing
1,sadness,Hi Everyone miss me much? muahhhhhhhhhhhhhhhh...
2,sadness,"rode the moped to the mall. fun stuff, its fli..."
3,sadness,gutted!!! vodafone wont repair my faulty Samsu...
4,happiness,"@shadowowns aww, &lt;3 why thank youu."


In [22]:
shuffled_df.shape

(10374, 2)

In [40]:
df_content = pd.DataFrame()

In [45]:
df_content['content'] = shuffled_df['content'].values

In [47]:
type(df_content)

pandas.DataFrame

In [32]:
df_sentiment = pd.DataFrame()

In [33]:
df_sentiment['sentiment'] = shuffled_df['sentiment'].map({"happiness":1, "sadness":0})

In [35]:
df_sentiment

,sentiment
0,0
1,0
2,0
3,0
4,1
...,...
10369,1
10370,0
10371,1
10372,0


In [ ]:
final_df = pd.concat([df_content, df_sentiment], axis=1)

In [51]:
final_df.head()

,content,sentiment
0,looks like we're rained out for weekend climbing,0
1,Hi Everyone miss me much? muahhhhhhhhhhhhhhhh...,0
2,"rode the moped to the mall. fun stuff, its fli...",0
3,gutted!!! vodafone wont repair my faulty Samsu...,0
4,"@shadowowns aww, &lt;3 why thank youu.",1


In [52]:
train_data, test_data = train_test_split(final_df, test_size=.2, random_state=42)

# Data Preprocessing

In [53]:
nltk.download("wordnet")
nltk.download("stopwords")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\as296\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\as296\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [62]:
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()

    text = text.split()

    text = [lemmatizer.lemmatize(y) for y in text]

    return " ".join(text)

In [69]:
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    text = [i for i in str(text).split() if i not in stop_words]

    return " ".join(text)

In [72]:
final_df['content'][2]

"rode the moped to the mall. fun stuff, its flippin gorgeous out. I'm sad that @maeannette is sick"

In [71]:
remove_stopwords(final_df['content'][2])

"rode moped mall. fun stuff, flippin gorgeous out. I'm sad @maeannette sick"

In [75]:
def remove_num(text):
    text = [i for i in text if not i.isdigit()]
    return "".join(text)

In [76]:
remove_num("hi, I am Asif 021")

'hi, I am Asif '

In [77]:
def lower_case(text):
    text = text.split()
    text = [y.lower() for y in text]

    return " ".join(text)

In [78]:
lower_case("hi I am ASIF")

'hi i am asif'

In [112]:
def remove_punctuatons(text):
    text = re.sub('[%s]' % re.escape("""`~!@#$%^&*()_-?+=.[\]{|}><,/:"""), ' ', text)
    text = text.replace(":","",)
    text = re.sub("\s+",' ', text)
    text = " ".join(text.split())

    return text.strip()


<>:2: SyntaxWarning: invalid escape sequence '\]'
<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\]'
<>:4: SyntaxWarning: invalid escape sequence '\s'
C:\Users\as296\AppData\Local\Temp\ipykernel_1436\2147044742.py:2: SyntaxWarning: invalid escape sequence '\]'
  text = re.sub('[%s]' % re.escape("""`~!@#$%^&*()_-?+=.[\]{|}><,/:"""), ' ', text)
C:\Users\as296\AppData\Local\Temp\ipykernel_1436\2147044742.py:4: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("\s+",' ', text)


In [85]:
remove_punctuatons("hi, asif.@ #$%^&/*()-_=/+")

'hi asif'

In [86]:
def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

In [87]:
remove_urls("this is a song https://www.youtube.com/watch?v=-63rM58hDAU&list=RD-63rM58hDAU&start_radio=1")

'this is a song '

In [105]:
def remove_small_sentences(df):
    for i in range(len(df)):
        if len(df.content.iloc[i].split()) <3:
            df.content.iloc[i] = np.nan

In [107]:
def normalize_text(df):
    df.content = df.content.apply(lambda content: lemmatization(content))
    df.content = df.content.apply(lambda content: remove_num(content))
    df.content = df.content.apply(lambda content: remove_punctuatons(content))
    df.content = df.content.apply(lambda content: remove_stopwords(content))
    df.content = df.content.apply(lambda content: remove_urls(content))
    df.content = df.content.apply(lambda content: lower_case(content))

    return df

In [110]:
def normalize_sentence(sentence):
    sentence = lower_case(sentence)
    sentence = remove_stopwords(sentence)
    sentence = remove_num(sentence)
    sentence = remove_punctuatons(sentence)
    sentence = remove_urls(sentence)
    sentence = lemmatization(sentence)
    return sentence

In [113]:
normalize_sentence("that's it? it's done already? This is one")

"that's it done already one"

In [115]:
train_data.head()

,content,sentiment
6192,@BabyPatches I had a very good day - lots of s...,1
2150,"@philritchie Boom, and if you will, boom! Saw ...",1
3118,@jaezors my b day is on may 13 but my party is...,1
8147,has just finished recording the improvisation ...,1
4949,I don't feel well. I feel like i could throw u...,0


In [116]:
train_data = normalize_text(train_data)
test_data = normalize_text(test_data)

In [117]:
train_data.head()

,content,sentiment
6192,babypatches i good day lot stretching sleeping...,1
2150,philritchie boom boom saw movie last night rea...,1
3118,jaezors b day may party sat come gay,1
8147,ha finished recording improvisation second ins...,1
4949,i feel well i feel like could throw throat hur...,0


# Feature Engineering

In [118]:
x_train = train_data['content'].values
x_test = test_data['content'].values

y_train = train_data['sentiment'].values
y_test = test_data['sentiment'].values

In [119]:
x_train.shape

(8299,)

In [120]:
x_test.shape

(2075,)

In [121]:
count_vectorizer = CountVectorizer()

x_train_bow = count_vectorizer.fit_transform(x_train)

x_test_bow = count_vectorizer.transform(x_test)

In [122]:
x_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 68435 stored elements and shape (8299, 14855)>

In [123]:
train_df = pd.DataFrame(x_train_bow.toarray())

train_df['label'] = y_train

In [124]:
train_df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,...,14816,14817,14818,14819,14820,14821,14822,14823,14824,14825,14826,14827,14828,14829,14830,14831,14832,14833,14834,14835,14836,14837,14838,14839,14840,14841,14842,14843,14844,14845,14846,14847,14848,14849,14850,14851,14852,14853,14854,label
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Model Building

In [131]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [128]:
model = XGBClassifier(use_label_encoder=False)

In [129]:
model.fit(x_train_bow, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


# Model Evaluation

In [130]:
y_pred = model.predict(x_test_bow)

In [132]:
accuracy = accuracy_score(y_test, y_pred=y_pred)
classification_report = classification_report(y_test, y_pred)

In [133]:
print(f"Accuracy: {accuracy}")
print(f"\n\nclassification report: {classification_report}")

Accuracy: 0.7633734939759036


classification report:               precision    recall  f1-score   support

           0       0.73      0.82      0.77      1021
           1       0.80      0.71      0.75      1054

    accuracy                           0.76      2075
   macro avg       0.77      0.76      0.76      2075
weighted avg       0.77      0.76      0.76      2075

